# Using GridSearchCV to find best model and perform hyper parameter tuning: Iris dataset

In [2]:
import pandas as pd

df = pd.read_csv("data_iris.csv")
df[47:130]

,sepal_length,sepal_width,petal_length,petal_width,species
47,4.6,3.2,1.4,0.2,setosa
48,5.3,3.7,1.5,0.2,setosa
49,5.0,3.3,1.4,0.2,setosa
50,7.0,3.2,4.7,1.4,versicolor
51,6.4,3.2,4.5,1.5,versicolor
...,...,...,...,...,...
125,7.2,3.2,6.0,1.8,virginica
126,6.2,2.8,4.8,1.8,virginica
127,6.1,3.0,4.9,1.8,virginica
128,6.4,2.8,5.6,2.1,virginica


## Approach 1 (BAD): Use train_test_split and manually tune parameters by trial and error

In [4]:
# Label encoding and data split ( Train + Test )
# step1: Separate features(X) and labels(y):
X = df[["sepal_length", "sepal_width", "petal_length", "petal_width"]]
y = df["species"]

print("X:\n", X)
print("y:\n", y)

X:
      sepal_length  sepal_width  petal_length  petal_width
0             5.1          3.5           1.4          0.2
1             4.9          3.0           1.4          0.2
2             4.7          3.2           1.3          0.2
3             4.6          3.1           1.5          0.2
4             5.0          3.6           1.4          0.2
..            ...          ...           ...          ...
145           6.7          3.0           5.2          2.3
146           6.3          2.5           5.0          1.9
147           6.5          3.0           5.2          2.0
148           6.2          3.4           5.4          2.3
149           5.9          3.0           5.1          1.8

[150 rows x 4 columns]
y:
 0         setosa
1         setosa
2         setosa
3         setosa
4         setosa
         ...    
145    virginica
146    virginica
147    virginica
148    virginica
149    virginica
Name: species, Length: 150, dtype: object


In [5]:
# step2: Encode labels BEFORE splitting
from sklearn.preprocessing  import LabelEncoder

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

print("Encoded labels:", y_encoded[:10]) # print 1st 10 values
print("Classes:", encoder.classes_)
print("Labels :", encoder.transform(encoder.classes_))

Encoded labels: [0 0 0 0 0 0 0 0 0 0]
Classes: ['setosa' 'versicolor' 'virginica']
Labels : [0 1 2]


In [8]:
# step3: Split the data into training and testing sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.3, shuffle=True)

In [9]:
print("Shape of training dataset:", X_train.shape) # 105 rows and 4 columns
print("Shape of testing dataset:", X_test.shape) # 105 rows and 4 columns

Shape of training dataset: (105, 4)
Shape of testing dataset: (45, 4)


In [14]:
from sklearn.svm import SVC

model = SVC(kernel='linear', # linear’, ‘poly’, ‘rbf’, ‘sigmoid’,
            C=50, 
            gamma='auto',
            degree = 4,
            decision_function_shape = 'ovr') # Number of combination = 4 kernels x 10 C x 2 gammas 
model.fit(X_train,y_train)
model.score(X_test, y_test)

0.9777777777777777

## Approach 2: Use K Fold Cross validation

**Manually try suppling models with different parameters to cross_val_score function with 5 fold cross validation**

In [15]:
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC
import numpy as np

In [30]:
score = cross_val_score(SVC(kernel='linear',C=1, gamma='auto'), X, y, cv=5)
print(score)
print(np.mean(score))

[0.96666667 1.         0.96666667 0.96666667 1.        ]
0.9800000000000001


In [31]:
score = cross_val_score(SVC(kernel='rbf',  C=10, gamma='auto'), X, y, cv=5) # C=1,10
print(score)
print(np.mean(score))

[0.96666667 1.         0.96666667 0.96666667 1.        ]
0.9800000000000001


In [28]:
score = cross_val_score(SVC(kernel='rbf', C=20, gamma='auto'), X, y, cv=5)
print(score)
print(np.mean(score))

[0.96666667 1.         0.9        0.96666667 1.        ]
0.9666666666666668


**From above results we can say that rbf with C=1 or 10 or linear with C=1 will give best performance**

## Approach 3: Use GridSearchCV
**GridSearchCV does exactly same thing as above but in a single line of code**

In [20]:
from sklearn.model_selection import GridSearchCV

clf = GridSearchCV(SVC(gamma='auto'), {
    'C': [1,10,20],
    'kernel': ['rbf','linear']
}, cv=5, return_train_score=False)

clf.fit(X, y)
clf.cv_results_

{'mean_fit_time': array([0.00347457, 0.00340204, 0.00539756, 0.0029037 , 0.00219264,
        0.00240388]),
 'std_fit_time': array([0.00086647, 0.00049375, 0.00242381, 0.00049523, 0.00039445,
        0.00049495]),
 'mean_score_time': array([0.00279312, 0.00299783, 0.00279441, 0.00200572, 0.002     ,
        0.00200014]),
 'std_score_time': array([7.32604056e-04, 1.54467968e-03, 3.95762018e-04, 6.74450721e-06,
        3.56832255e-07, 1.22940789e-05]),
 'param_C': masked_array(data=[1, 1, 10, 10, 20, 20],
              mask=[False, False, False, False, False, False],
        fill_value='?',
             dtype=object),
 'param_kernel': masked_array(data=['rbf', 'linear', 'rbf', 'linear', 'rbf', 'linear'],
              mask=[False, False, False, False, False, False],
        fill_value='?',
             dtype=object),
 'params': [{'C': 1, 'kernel': 'rbf'},
  {'C': 1, 'kernel': 'linear'},
  {'C': 10, 'kernel': 'rbf'},
  {'C': 10, 'kernel': 'linear'},
  {'C': 20, 'kernel': 'rbf'},
  {'C': 20

In [21]:
df = pd.DataFrame(clf.cv_results_)
df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_kernel,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.003475,0.000866,0.002793,7.326041e-04,1,rbf,"{'C': 1, 'kernel': 'rbf'}",0.966667,1.0,0.966667,0.966667,1.0,0.980000,0.016330,1
1,0.003402,0.000494,0.002998,1.544680e-03,1,linear,"{'C': 1, 'kernel': 'linear'}",0.966667,1.0,0.966667,0.966667,1.0,0.980000,0.016330,1
2,0.005398,0.002424,0.002794,3.957620e-04,10,rbf,"{'C': 10, 'kernel': 'rbf'}",0.966667,1.0,0.966667,0.966667,1.0,0.980000,0.016330,1
3,0.002904,0.000495,0.002006,6.744507e-06,10,linear,"{'C': 10, 'kernel': 'linear'}",1.000000,1.0,0.900000,0.966667,1.0,0.973333,0.038873,4
4,0.002193,0.000394,0.002000,3.568323e-07,20,rbf,"{'C': 20, 'kernel': 'rbf'}",0.966667,1.0,0.900000,0.966667,1.0,0.966667,0.036515,5
5,0.002404,0.000495,0.002000,1.229408e-05,20,linear,"{'C': 20, 'kernel': 'linear'}",1.000000,1.0,0.900000,0.933333,1.0,0.966667,0.042164,6


In [30]:
df[['param_C','param_kernel','mean_test_score']]

,param_C,param_kernel,mean_test_score
0,1,rbf,0.980000
1,1,linear,0.980000
2,10,rbf,0.980000
3,10,linear,0.973333
4,20,rbf,0.966667
5,20,linear,0.966667


In [31]:
clf.best_params_

{'C': 1, 'kernel': 'rbf'}

In [32]:
clf.best_score_

0.9800000000000001

## Now try different models with different hyperparameters

In [22]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

model_params = {
    'svm': {
        'model': SVC(gamma='auto'),
        'params' : {
            'C': [1,10,20],
            'kernel': ['rbf','linear']
        }  
    },
    'random_forest': {
        'model': RandomForestClassifier(),
        'params' : {
            'n_estimators': [1,5,10]
        }
    },
    'logistic_regression' : {
        'model': LogisticRegression(solver='liblinear', multi_class='auto'),
        'params': {
            'C': [1,5,10]
        }
    }
}


In [23]:
scores = []

for model_name, mp in model_params.items():
    clf =  GridSearchCV(mp['model'], mp['params'], cv=5, return_train_score=False)
    clf.fit(X, y)
    scores.append({
        'model': model_name,
        'best_score': clf.best_score_,
        'best_params': clf.best_params_
    })
    
df = pd.DataFrame(scores,columns=['model','best_score','best_params'])
df

,model,best_score,best_params
0,svm,0.980000,"{'C': 1, 'kernel': 'rbf'}"
1,random_forest,0.960000,{'n_estimators': 1}
2,logistic_regression,0.966667,{'C': 5}


### Conclusion: SVM with C=1 and kernel='rbf' is the best model

## SIDE NOTE
**Use RandomizedSearchCV to reduce number of iterations and with random combination of parameters. This is useful when you have too many parameters to try and your training time is longer. It helps reduce the cost of computation**

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

rs = RandomizedSearchCV(SVC(gamma='auto'), {
        'C': [1,10,20],
        'kernel': ['rbf','linear']
    }, 
    cv=5, 
    return_train_score=False, 
    n_iter=2
)
rs.fit(iris.data, iris.target)
pd.DataFrame(rs.cv_results_)[['param_C','param_kernel','mean_test_score']]